# Contextual charts

This notebook pulls local-election result data from `source_data/election_results` and charts the average number of candidates per seat over time.

The House of Commons Library files support this metric for the handbook datasets from 2021 onward. The earlier 2016-2019 result-analysis files are council-level summaries and do not contain the candidate-level or ward-level seat counts needed for this calculation.

Metric definition: total candidates divided by total seats contested in each election year.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

In [2]:
def resolve_repo_root() -> Path:
    for base in (Path.cwd(), *Path.cwd().parents):
        if (base / "source_data").exists():
            return base
    raise FileNotFoundError("Could not locate repo root from the current working directory.")


REPO_ROOT = resolve_repo_root()
LOCAL_ELECTIONS_DIR = REPO_ROOT / "source_data" / "election_results" / "local_elections"
NOTEBOOK_DIR = REPO_ROOT / "labour_lessons_2026_elections" / "contextual_analysis"
CHART_PATH = NOTEBOOK_DIR / "average_candidates_per_seat_by_local_election.png"
SUMMARY_PATH = NOTEBOOK_DIR / "average_candidates_per_seat_by_local_election.csv"

YEAR_CONFIGS = [
    {
        "year": 2021,
        "path": LOCAL_ELECTIONS_DIR / "2021_handbook_dataset" / "local_elections_2021_results-2.xlsx",
        "candidate_sheet": "Candidates-results",
        "candidate_header": 1,
        "candidate_name_col": "Candidate name",
        "ward_sheet": "Wards-results",
        "ward_header": 1,
        "seat_col": "Vacancies",
        "ward_key_cols": ["Local authority code", "Ward/ED code", "Ward/ED name"],
    },
    {
        "year": 2022,
        "path": LOCAL_ELECTIONS_DIR / "2022_handbook_dataset" / "local-elections-2022.xlsx",
        "candidate_sheet": "Candidates-results",
        "candidate_header": 1,
        "candidate_name_col": "Candidate name",
        "ward_sheet": "Wards-results",
        "ward_header": 1,
        "seat_col": "Vacancies",
        "ward_key_cols": ["Local authority code", "Ward code", "Ward name"],
    },
    {
        "year": 2023,
        "path": LOCAL_ELECTIONS_DIR / "2023_handbook_dataset" / "LEH-Candidates-2023.xlsx",
        "candidate_sheet": "Cand_Table",
        "candidate_header": 0,
        "candidate_name_col": "NAME",
        "ward_sheet": "Ward_Level",
        "ward_header": 0,
        "seat_col": "VACS",
        "ward_key_cols": ["DISTRICTNAME", "WARDNAME"],
    },
    {
        "year": 2024,
        "path": LOCAL_ELECTIONS_DIR / "2024_handbook_dataset" / "LEH-2024-results-HoC-version.xlsx",
        "candidate_sheet": "Candidates results",
        "candidate_header": 1,
        "candidate_name_col": "Name",
        "ward_sheet": "Wards results",
        "ward_header": 1,
        "seat_col": "Vacancies",
        "ward_key_cols": ["Local authority code", "Ward code", "Ward name"],
    },
    {
        "year": 2025,
        "path": LOCAL_ELECTIONS_DIR / "2025_handbook_dataset" / "LEH-2025-results-HoC.xlsx",
        "candidate_sheet": "Candidates result",
        "candidate_header": 1,
        "candidate_name_col": "Candidate name",
        "ward_sheet": "Ward results",
        "ward_header": 1,
        "seat_col": "Seats",
        "ward_key_cols": ["ONS ward code", "Ward/ County Electoral District name"],
    },
]

In [3]:
def read_excel_clean(path: Path, sheet_name: str, header: int) -> pd.DataFrame:
    df = pd.read_excel(path, sheet_name=sheet_name, header=header)
    df.columns = [str(col).strip() for col in df.columns]
    return df


def load_local_election_summary(config: dict) -> dict:
    candidates = read_excel_clean(
        config["path"],
        sheet_name=config["candidate_sheet"],
        header=config["candidate_header"],
    )
    wards = read_excel_clean(
        config["path"],
        sheet_name=config["ward_sheet"],
        header=config["ward_header"],
    )

    candidate_rows = candidates[candidates[config["candidate_name_col"]].notna()].copy()
    ward_rows = wards[wards[config["seat_col"]].notna()].copy()
    ward_rows = ward_rows.drop_duplicates(subset=config["ward_key_cols"])

    total_candidates = int(candidate_rows.shape[0])
    total_seats = int(pd.to_numeric(ward_rows[config["seat_col"]], errors="coerce").fillna(0).sum())

    return {
        "year": config["year"],
        "total_candidates": total_candidates,
        "total_seats": total_seats,
        "avg_candidates_per_seat": total_candidates / total_seats,
    }


summary = pd.DataFrame(load_local_election_summary(config) for config in YEAR_CONFIGS)
summary = summary.sort_values("year").reset_index(drop=True)
summary_display = summary.copy()
summary_display["avg_candidates_per_seat"] = summary_display["avg_candidates_per_seat"].round(2)

display(summary_display)

summary.to_csv(SUMMARY_PATH, index=False)

,year,total_candidates,total_seats,avg_candidates_per_seat
0,2021,18044,4658,3.87
1,2022,18481,5555,3.33
2,2023,25697,8031,3.20
3,2024,10029,2659,3.77
4,2025,8141,1641,4.96


In [4]:
sns.set_theme(style="white")

fig, ax = plt.subplots(figsize=(10, 6.5))
bars = ax.bar(
    summary["year"].astype(str),
    summary["avg_candidates_per_seat"],
    color="#c8102e",
    width=0.7,
)

fig.suptitle(
    "Average number of candidates per seat in local elections",
    x=0.01,
    y=0.98,
    ha="left",
    fontsize=18,
    fontweight="bold",
)
fig.text(
    0.01,
    0.92,
    "House of Commons Library handbook datasets, 2021-2025",
    ha="left",
    va="top",
    fontsize=12,
    color="#555555",
)
ax.set_ylabel("Candidates per seat", fontsize=16, fontweight="bold")
ax.set_xlabel("")
ax.set_ylim(0, summary["avg_candidates_per_seat"].max() * 1.18)
ax.tick_params(axis="both", labelsize=14, length=0)
for label in ax.get_xticklabels() + ax.get_yticklabels():
    label.set_fontweight("bold")
ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

for bar, value in zip(bars, summary["avg_candidates_per_seat"]):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        value + 0.05,
        f"{value:.2f}",
        ha="center",
        va="bottom",
        fontsize=14,
        fontweight="bold",
    )

fig.text(
    0.01,
    0.01,
    "Source: House of Commons Library local election handbook datasets in source_data/election_results.",
    ha="left",
    fontsize=9,
    color="#555555",
)

plt.tight_layout(rect=[0, 0.05, 1, 0.86])
fig.savefig(CHART_PATH, dpi=200, bbox_inches="tight")
plt.close(fig)
print(f"Saved chart to {CHART_PATH}")

Saved chart to C:\Users\DamayantiChatterjee\GitHub\uk-politics\labour_lessons_2026_elections\average_candidates_per_seat_by_local_election.png


## 2026 contesting wards mapped to their last result

This section maps the wards that were contesting elections in 2026, coloured by the party that won each ward at its **most recent previous local election**.

Matching logic:
- Each 2026 contest (from the electionresults.uk race list) is matched by council + ward name to the latest official ONS ward geometry, `Wards (December 2025) Boundaries UK BGC` (already in British National Grid metres, so the map plots in the correct projection).
- The matched ONS ward code is then looked up against the House of Commons Library local-election handbook datasets (2021-2025) to find the most recent year the ward was contested, and the winning party in that year (the party with the most ward votes; 2023 is matched by name as that file carries no ONS codes).

To make the map recognisable, every GB ward is drawn in light grey as a background, with the contesting wards coloured on top.

Two groups of 2026 contests cannot be placed on this map and are reported as unmatched:
- **County council elections** (e.g. Norfolk, Essex, Hampshire, the Sussexes, Suffolk, Surrey) run on *county electoral divisions*, which use entirely different geometry that is not in the ward boundary file.
- **2026 boundary-review wards** (e.g. parts of Swindon, Sunderland, Sefton, Milton Keynes) were fought on new ward names whose geometry the ONS had not yet published as of the snapshot date.

In [5]:
import json
import io
import re

import numpy as np
import requests
from matplotlib.collections import PatchCollection
from matplotlib.patches import Patch, Polygon

REQUEST_HEADERS = {"User-Agent": "Mozilla/5.0"}
GEOGRAPHY_DIR = REPO_ROOT / "source_data" / "geography"
GEOGRAPHY_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_ELECTIONS_2026_DIR = LOCAL_ELECTIONS_DIR / "2026_external"
LOCAL_ELECTIONS_2026_DIR.mkdir(parents=True, exist_ok=True)

WARD_BOUNDARY_URL = "https://open-geography-portalx-ons.hub.arcgis.com/api/download/v1/items/5cc0a7e3aa194a1080026eb13ce48dbe/geojson?layers=0"
RACES_URL = "https://electionresults.uk/data/races.csv"
COUNCIL_CONTROL_URL = "https://opencouncildata.co.uk/history2016-26.csv"

WARD_BOUNDARY_PATH = GEOGRAPHY_DIR / "wards_december_2025_boundaries_uk_bgc.geojson"
RACES_CACHE_PATH = LOCAL_ELECTIONS_2026_DIR / "electionresults_uk_races.csv"
COUNCIL_CONTROL_CACHE_PATH = LOCAL_ELECTIONS_2026_DIR / "opencouncildata_history2016_26.csv"

WARD_RESULT_MATCHED_PATH = NOTEBOOK_DIR / "contesting_wards_2026_by_last_result_matched.csv"
WARD_RESULT_UNMATCHED_PATH = NOTEBOOK_DIR / "contesting_wards_2026_by_last_result_unmatched.csv"
WARD_RESULT_MAP_PATH = NOTEBOOK_DIR / "contesting_wards_2026_by_last_result.png"


def download_text_if_missing(url: str, path: Path) -> Path:
    if path.exists():
        return path
    response = requests.get(url, headers=REQUEST_HEADERS, timeout=300)
    response.raise_for_status()
    path.write_text(response.text, encoding="utf-8")
    return path


def normalise_name(value: str) -> str:
    value = str(value).strip().lower().replace("&", "and")
    value = re.sub(r"[^a-z0-9]+", " ", value)
    return re.sub(r"\s+", " ", value).strip()


def normalise_council(value: str) -> str:
    """Council names, dropping ', City of' / ', County of' style suffixes before normalising."""
    value = re.sub(r",?\s*(city of|county of|borough of|district)\b", "", str(value), flags=re.I)
    return normalise_name(value)


# Map handbook party codes to the display groups used on the map.
PARTY_GROUPS = {
    "LAB": "Labour",
    "CON": "Conservative",
    "LD": "Liberal Democrat",
    "GREEN": "Green",
    "GRN": "Green",
    "REF": "Reform UK",
    "IND": "Independent",
    "PC": "Plaid Cymru",
    "SNP": "SNP",
}


def party_group(code: str) -> str:
    return PARTY_GROUPS.get(str(code).strip().upper(), "Other")


def simplify_control_label(value: str) -> str:
    """Collapse an Open Council Data 'majority' label (e.g. 'LAB min', 'LD/IND') into a display group."""
    value = str(value).strip()
    if not value or value.lower() == "nan":
        return "Other"
    if value == "TBC":
        return "TBC"
    if "/" in value:
        return "Coalition"
    token = value.split()[0].upper()
    token_map = {
        "CON": "Conservative",
        "LAB": "Labour",
        "LD": "Liberal Democrat",
        "GRN": "Green",
        "GREEN": "Green",
        "REF": "Reform UK",
        "SNP": "SNP",
        "PC": "Plaid Cymru",
        "IND": "Independent",
        "ASPIRE": "Aspire",
    }
    return token_map.get(token, "Other")


# Columns in the handbook ward sheets that are metadata, not party vote tallies.
META_TOKENS = [
    "code", "name", "county", "authority", "vacanc", "seat", "type", "electorate",
    "turnout", "ballot", "invalid", "total", "grand", "ward", "district", "region",
    "constituency", "upper", "lower", "ons", "ec ", "elect", "other parties",
]


def winning_party_column(df: pd.DataFrame) -> pd.Series:
    """Per row, return the party column with the most votes (the ward winner)."""
    party_cols = [c for c in df.columns if not any(tok in str(c).strip().lower() for tok in META_TOKENS)]
    votes = df[party_cols].apply(pd.to_numeric, errors="coerce")
    votes = votes.dropna(axis=1, how="all")
    votes = votes[[c for c in votes.columns if votes[c].fillna(0).sum() > 0]]
    return votes.idxmax(axis=1)


# Handbook ward-result sheets that carry an ONS ward code, newest first.
WARD_RESULT_CONFIGS = [
    {"year": 2025, "path": LOCAL_ELECTIONS_DIR / "2025_handbook_dataset" / "LEH-2025-results-HoC.xlsx",
     "sheet": "Ward results", "header": 1, "code_col": "ONS ward code"},
    {"year": 2024, "path": LOCAL_ELECTIONS_DIR / "2024_handbook_dataset" / "LEH-2024-results-HoC-version.xlsx",
     "sheet": "Wards results", "header": 1, "code_col": "Ward code"},
    {"year": 2022, "path": LOCAL_ELECTIONS_DIR / "2022_handbook_dataset" / "local-elections-2022.xlsx",
     "sheet": "Wards-results", "header": 1, "code_col": "Ward code"},
    {"year": 2021, "path": LOCAL_ELECTIONS_DIR / "2021_handbook_dataset" / "local_elections_2021_results-2.xlsx",
     "sheet": "Wards-results", "header": 1, "code_col": "Ward/ED code"},
]

# Newest-first list of years searched to find each ward's most recent prior result.
WINNER_BY_CODE = {}
for cfg in WARD_RESULT_CONFIGS:
    sheet = read_excel_clean(cfg["path"], sheet_name=cfg["sheet"], header=cfg["header"])
    sheet = sheet[sheet[cfg["code_col"]].notna()].copy()
    sheet["winner"] = winning_party_column(sheet)
    sheet = sheet[sheet["winner"].notna()]
    WINNER_BY_CODE[cfg["year"]] = {
        str(code).strip(): party_group(winner)
        for code, winner in zip(sheet[cfg["code_col"]], sheet["winner"])
    }

# 2023 handbook carries no ONS code, so it is matched by council + ward name as a fallback.
sheet_2023 = read_excel_clean(
    LOCAL_ELECTIONS_DIR / "2023_handbook_dataset" / "LEH-Candidates-2023.xlsx",
    sheet_name="Ward_Level", header=0,
)
sheet_2023["winner"] = winning_party_column(sheet_2023)
sheet_2023 = sheet_2023[sheet_2023["winner"].notna()]
WINNER_BY_NAME_2023 = {
    (normalise_council(d), normalise_name(w)): party_group(winner)
    for d, w, winner in zip(sheet_2023["DISTRICTNAME"], sheet_2023["WARDNAME"], sheet_2023["winner"])
}

SEARCH_ORDER = [2025, 2024, 2022, 2021]

download_text_if_missing(WARD_BOUNDARY_URL, WARD_BOUNDARY_PATH)
download_text_if_missing(RACES_URL, RACES_CACHE_PATH)
download_text_if_missing(COUNCIL_CONTROL_URL, COUNCIL_CONTROL_CACHE_PATH)

ward_geojson = json.loads(WARD_BOUNDARY_PATH.read_text(encoding="utf-8"))
boundary_features = ward_geojson["features"]

ward_lookup = pd.DataFrame(
    {
        "boundary_code": feature["properties"]["WD25CD"],
        "ward_key": normalise_name(feature["properties"]["WD25NM"]),
        "council_key": normalise_council(feature["properties"]["LAD25NM"]),
        "feature_index": feature_index,
    }
    for feature_index, feature in enumerate(boundary_features)
)

races = pd.read_csv(RACES_CACHE_PATH)
races_2026 = races.loc[races["year"] == 2026, ["council", "ward_name", "seats"]].drop_duplicates().copy()
races_2026["council_key"] = races_2026["council"].map(normalise_council)
races_2026["ward_key"] = races_2026["ward_name"].map(normalise_name)

# 2026 council control (Open Council Data): keeps the raw 'majority' label plus a simplified 'control_group'.
council_control = pd.read_csv(COUNCIL_CONTROL_CACHE_PATH)
council_control.columns = [str(column).strip() for column in council_control.columns]
council_control_2026 = (
    council_control.loc[council_control["year"] == 2026, ["authority", "majority"]].drop_duplicates().copy()
)
council_control_2026["council_key"] = council_control_2026["authority"].map(normalise_council)
council_control_2026["control_group"] = council_control_2026["majority"].map(simplify_control_label)

contesting_wards_2026 = races_2026.merge(
    council_control_2026[["council_key", "majority", "control_group"]],
    on="council_key",
    how="left",
).merge(
    ward_lookup[["council_key", "ward_key", "boundary_code", "feature_index"]],
    on=["council_key", "ward_key"],
    how="left",
).drop_duplicates(subset=["council", "ward_name"])


def last_result(row: pd.Series) -> pd.Series:
    code = row["boundary_code"]
    if pd.notna(code):
        for year in SEARCH_ORDER:
            party = WINNER_BY_CODE[year].get(str(code).strip())
            if party is not None:
                return pd.Series([party, year])
    party = WINNER_BY_NAME_2023.get((row["council_key"], row["ward_key"]))
    if party is not None:
        return pd.Series([party, 2023])
    return pd.Series([None, None])


contesting_wards_2026[["last_result_party", "last_result_year"]] = contesting_wards_2026.apply(last_result, axis=1)

matched_wards_2026 = contesting_wards_2026[contesting_wards_2026["boundary_code"].notna()].copy()
matched_wards_2026 = matched_wards_2026.drop_duplicates(subset=["boundary_code"])
unmatched_wards_2026 = contesting_wards_2026[contesting_wards_2026["boundary_code"].isna()].copy()

matched_wards_2026.to_csv(WARD_RESULT_MATCHED_PATH, index=False)
unmatched_wards_2026.to_csv(WARD_RESULT_UNMATCHED_PATH, index=False)

coverage_summary = pd.DataFrame(
    [
        {
            "total_2026_contesting_wards": int(len(races_2026)),
            "matched_to_ward_geometry": int(len(matched_wards_2026)),
            "of_which_with_last_result": int(matched_wards_2026["last_result_party"].notna().sum()),
            "unmatched": int(len(unmatched_wards_2026)),
            "match_rate": round(len(matched_wards_2026) / len(races_2026), 3),
        }
    ]
)
display(coverage_summary)

unmatched_by_council = (
    unmatched_wards_2026.groupby("council").size().reset_index(name="unmatched_wards").sort_values("unmatched_wards", ascending=False)
)
display(unmatched_by_council.head(15))

C:\Users\DamayantiChatterjee\AppData\Local\Temp\claude\ipykernel_29036\3479666409.py:139: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  sheet_2023["winner"] = winning_party_column(sheet_2023)


,total_2026_contesting_wards,matched_to_ward_geometry,of_which_with_last_result,unmatched,match_rate
0,2943,2305,2301,638,0.783


,council,unmatched_wards
12,Norfolk,84
6,Essex,77
8,Hampshire,76
24,West Sussex,70
17,Suffolk,69
5,East Sussex,50
23,West Surrey,45
4,East Surrey,36
19,Swindon,21
18,Sunderland,16


In [6]:
PARTY_COLORS = {
    "Labour": "#E4003B",
    "Conservative": "#0087DC",
    "Liberal Democrat": "#FAA61A",
    "Green": "#6AB023",
    "Reform UK": "#12B6CF",
    "SNP": "#FDF38E",
    "Plaid Cymru": "#3F8428",
    "Independent": "#7F8C8D",
    "Other": "#B0A8B9",
    "No prior result": "#D9D9D9",
}

BACKGROUND_FILL = "#EFEFEF"
BACKGROUND_EDGE = "#FFFFFF"

legend_order = [
    "Labour",
    "Conservative",
    "Liberal Democrat",
    "Reform UK",
    "Green",
    "Independent",
    "Other",
    "No prior result",
]


def geometry_to_patches(geometry: dict) -> list[Polygon]:
    geometry_type = geometry.get("type")
    if geometry_type == "Polygon":
        polygon_sets = [geometry["coordinates"]]
    elif geometry_type == "MultiPolygon":
        polygon_sets = geometry["coordinates"]
    else:
        return []
    return [Polygon(np.asarray(polygon[0]), closed=True) for polygon in polygon_sets if polygon]


matched_wards_2026["plot_group"] = matched_wards_2026["last_result_party"].fillna("No prior result")

fig, ax = plt.subplots(figsize=(11, 14))

# Light-grey backdrop of every English ward so the contesting wards read against a recognisable map.
# All 2026 council contests are in England, so the map is cropped to England (ward codes starting with "E").
english_features = [feature for feature in boundary_features if str(feature["properties"]["WD25CD"]).startswith("E")]
background_patches = [patch for feature in english_features for patch in geometry_to_patches(feature["geometry"])]
ax.add_collection(
    PatchCollection(background_patches, facecolor=BACKGROUND_FILL, edgecolor=BACKGROUND_EDGE, linewidths=0.05, zorder=1)
)

# Contesting wards coloured by the winner of their most recent previous local election.
patches_by_group = {group: [] for group in legend_order}
for row in matched_wards_2026.itertuples(index=False):
    geometry = boundary_features[int(row.feature_index)]["geometry"]
    patches_by_group.setdefault(row.plot_group, []).extend(geometry_to_patches(geometry))

for group in legend_order:
    patches = patches_by_group.get(group, [])
    if not patches:
        continue
    ax.add_collection(
        PatchCollection(
            patches,
            facecolor=PARTY_COLORS.get(group, PARTY_COLORS["Other"]),
            edgecolor="white",
            linewidths=0.1,
            zorder=2,
        )
    )

ax.autoscale_view()
ax.set_aspect("equal")
ax.axis("off")

fig.suptitle(
    "How England's 2026 contesting wards last voted",
    x=0.01,
    y=0.985,
    ha="left",
    fontsize=18,
    fontweight="bold",
)
fig.text(
    0.01,
    0.955,
    f"Each ward contesting in 2026 coloured by the party that won it at its most recent previous local election. "
    f"Mapped {len(matched_wards_2026):,} of {len(races_2026):,} 2026 contesting wards "
    f"({len(matched_wards_2026) / len(races_2026):.0%}); county-division and 2026 boundary-change contests are not shown.",
    ha="left",
    va="top",
    fontsize=11,
    color="#555555",
    wrap=True,
)

legend_handles = [
    Patch(facecolor=PARTY_COLORS[group], edgecolor="none", label=group)
    for group in legend_order
    if patches_by_group.get(group)
]
ax.legend(
    handles=legend_handles,
    loc="lower left",
    bbox_to_anchor=(0.01, -0.02),
    ncol=2,
    frameon=False,
    fontsize=11,
)

fig.text(
    0.01,
    0.015,
    "Sources: electionresults.uk 2026 race list; House of Commons Library local-election handbook datasets 2021-2025 (ward winners); "
    "ONS Wards (December 2025) Boundaries UK BGC. County-council divisions and 2026 boundary-review wards are omitted where current ward geometry is unavailable.",
    ha="left",
    fontsize=8.5,
    color="#555555",
    wrap=True,
)

plt.tight_layout(rect=[0, 0.04, 1, 0.93])
fig.savefig(WARD_RESULT_MAP_PATH, dpi=200, bbox_inches="tight")
plt.close(fig)
print(f"Saved chart to {WARD_RESULT_MAP_PATH}")

Saved chart to C:\Users\DamayantiChatterjee\GitHub\uk-politics\labour_lessons_2026_elections\contesting_wards_2026_by_last_result.png


## Are the 2026 contesting wards younger and more urban than England as a whole?

This section compares the wards contesting in 2026 against all English wards on two Census 2021 measures:

- **Age** - the median age of usual residents in each ward (computed from single-year-of-age counts, ONS table TS007).
- **Urbanity** - population density, usual residents per square kilometre (ONS table TS006), used here as a continuous proxy for how urban a ward is.

Both come from the 2021 Census on 2022 ward boundaries (NOMIS geography type TYPE153). Contesting wards are matched to the census by ONS ward code; wards whose 2026 / December-2025 code differs from a 2022 census ward (boundary changes) drop out and the match rate is reported.

In [7]:
import io

CENSUS_DIR = REPO_ROOT / "source_data" / "census_2021"
CENSUS_DIR.mkdir(parents=True, exist_ok=True)

NOMIS_BASE = "https://www.nomisweb.co.uk/api/v01/dataset/"
WARD_TYPE = "TYPE153"  # 2022 wards
DENSITY_CACHE_PATH = CENSUS_DIR / "ts006_population_density_2022wards.csv"
AGE_CACHE_PATH = CENSUS_DIR / "ts007_single_year_age_2022wards.csv"

AGE_URBAN_SUMMARY_PATH = NOTEBOOK_DIR / "contesting_wards_2026_age_urbanity_summary.csv"


def fetch_nomis_density() -> pd.DataFrame:
    if not DENSITY_CACHE_PATH.exists():
        url = (
            f"{NOMIS_BASE}NM_2026_1.data.csv?geography={WARD_TYPE}"
            "&cell=0&measures=20100&select=GEOGRAPHY_CODE,OBS_VALUE"
        )
        DENSITY_CACHE_PATH.write_text(requests.get(url, headers=REQUEST_HEADERS, timeout=300).text, encoding="utf-8")
    density = pd.read_csv(DENSITY_CACHE_PATH)
    density.columns = ["ward_code", "density"]
    return density


def fetch_nomis_single_year_age() -> pd.DataFrame:
    # Single-year-of-age counts (TS007). NOMIS caps guest queries at 25,000 rows
    # (~7,600 wards per age), so the 101 age codes are pulled three at a time.
    if not AGE_CACHE_PATH.exists():
        frames = []
        age_codes = list(range(1, 102))  # code 1 = age 0 ... code 101 = age 100+
        for start in range(0, len(age_codes), 3):
            batch = ",".join(str(code) for code in age_codes[start:start + 3])
            url = (
                f"{NOMIS_BASE}NM_2027_1.data.csv?geography={WARD_TYPE}"
                f"&c2021_age_102={batch}&measures=20100&select=GEOGRAPHY_CODE,C2021_AGE_102,OBS_VALUE"
            )
            frames.append(pd.read_csv(io.StringIO(requests.get(url, headers=REQUEST_HEADERS, timeout=120).text)))
        age = pd.concat(frames, ignore_index=True)
        age.columns = ["ward_code", "age_code", "population"]
        age.to_csv(AGE_CACHE_PATH, index=False)
    age = pd.read_csv(AGE_CACHE_PATH)
    age["age"] = age["age_code"] - 1  # code 1 -> age 0
    return age


def ward_median_age(group: pd.DataFrame) -> float:
    group = group.sort_values("age")
    total = group["population"].sum()
    if total <= 0:
        return np.nan
    cumulative = 0
    for age, population in zip(group["age"], group["population"]):
        if cumulative + population >= total / 2:
            return age + (total / 2 - cumulative) / population if population else age
        cumulative += population
    return np.nan


density = fetch_nomis_density()
age_counts = fetch_nomis_single_year_age()
median_age = (
    age_counts.groupby("ward_code").apply(ward_median_age, include_groups=False).rename("median_age").reset_index()
)

# One row per English ward with both measures, flagged for whether it is contesting in 2026.
contested_codes = set(matched_wards_2026.loc[matched_wards_2026["boundary_code"].notna(), "boundary_code"])
ward_profile = density.merge(median_age, on="ward_code", how="inner")
ward_profile = ward_profile[ward_profile["ward_code"].str.startswith("E")].copy()
ward_profile["contesting_2026"] = ward_profile["ward_code"].isin(contested_codes)

contested_english = {code for code in contested_codes if str(code).startswith("E")}
matched_to_census = ward_profile["contesting_2026"].sum()

coverage_note = pd.DataFrame(
    [
        {
            "contesting_english_wards_on_map": len(contested_english),
            "matched_to_census_2021": int(matched_to_census),
            "census_match_rate": round(matched_to_census / len(contested_english), 3),
        }
    ]
)
display(coverage_note)


def describe(frame: pd.DataFrame) -> dict:
    return {
        "wards": int(len(frame)),
        "median_age": round(frame["median_age"].median(), 1),
        "median_density_per_km2": int(round(frame["density"].median())),
        "share_density_over_4000": round((frame["density"] > 4000).mean(), 3),
    }


comparison = pd.DataFrame(
    {
        "All England wards": describe(ward_profile),
        "2026 contesting wards": describe(ward_profile[ward_profile["contesting_2026"]]),
        "Not contesting in 2026": describe(ward_profile[~ward_profile["contesting_2026"]]),
    }
).T
comparison.to_csv(AGE_URBAN_SUMMARY_PATH)
display(comparison)

,contesting_english_wards_on_map,matched_to_census_2021,census_match_rate
0,2305,1889,0.82


,wards,median_age,median_density_per_km2,share_density_over_4000
All England wards,6876.0,43.3,1844.0,0.239
2026 contesting wards,1889.0,38.3,4069.0,0.507
Not contesting in 2026,4987.0,45.4,1107.0,0.138


In [8]:
AGE_URBAN_CHART_PATH = NOTEBOOK_DIR / "contesting_wards_2026_age_urbanity.png"

GROUP_ORDER = ["2026 contesting wards", "All England wards"]
GROUP_COLORS = {"2026 contesting wards": "#E4003B", "All England wards": "#B0B0B0"}

plot_data = pd.concat(
    [
        ward_profile.assign(group="All England wards"),
        ward_profile[ward_profile["contesting_2026"]].assign(group="2026 contesting wards"),
    ]
)

sns.set_theme(style="white")
fig, axes = plt.subplots(1, 2, figsize=(12, 6.5))

for ax, column, title, ylabel, log in [
    (axes[0], "median_age", "Median age of ward", "Years", False),
    (axes[1], "density", "Population density", "Usual residents per km² (log scale)", True),
]:
    sns.boxplot(
        data=plot_data,
        x="group",
        y=column,
        hue="group",
        order=GROUP_ORDER,
        palette=GROUP_COLORS,
        legend=False,
        width=0.6,
        fliersize=1,
        ax=ax,
    )
    if log:
        ax.set_yscale("log")
    medians = plot_data.groupby("group")[column].median()
    for i, group in enumerate(GROUP_ORDER):
        ax.annotate(
            f"{medians[group]:,.0f}" if log else f"{medians[group]:.1f}",
            xy=(i, medians[group]),
            ha="center",
            va="bottom",
            fontsize=11,
            fontweight="bold",
        )
    ax.set_title(title, fontsize=14, fontweight="bold")
    ax.set_xlabel("")
    ax.set_ylabel(ylabel, fontsize=12)
    ax.tick_params(axis="both", labelsize=11)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

fig.suptitle(
    "2026 contesting wards are younger and more urban than England as a whole",
    x=0.01,
    y=0.99,
    ha="left",
    fontsize=16,
    fontweight="bold",
)
fig.text(
    0.01,
    0.94,
    f"Census 2021 (ONS tables TS006 and TS007), {int(matched_to_census):,} contesting wards matched to census ward geography. "
    "Box shows the interquartile range; line is the median.",
    ha="left",
    va="top",
    fontsize=10,
    color="#555555",
)
fig.text(
    0.01,
    0.01,
    "Sources: ONS Census 2021 via NOMIS (TS006 population density; TS007 age by single year of age), 2022 ward geography.",
    ha="left",
    fontsize=8.5,
    color="#555555",
)

plt.tight_layout(rect=[0, 0.03, 1, 0.9])
fig.savefig(AGE_URBAN_CHART_PATH, dpi=200, bbox_inches="tight")
plt.close(fig)
print(f"Saved chart to {AGE_URBAN_CHART_PATH}")

Saved chart to C:\Users\DamayantiChatterjee\GitHub\uk-politics\labour_lessons_2026_elections\contesting_wards_2026_age_urbanity.png


## Governing parties vs their polling average at local elections

This recreates the Stack Data Strategy chart: for every local-election year since 1982, the **governing party's Projected National Share (PNS)** minus its **national polling average going into the election**. A negative bar means the party in power did worse in the local elections than the polls implied.

Data sources:
- **PNS** — the BBC Projected National Share for the governing party, from Mark Pack's *LocalBase* (the companion dataset to PollBase; cached to `source_data/polling/`). BBC PNS runs from 1982, which is why the series starts there.
- **Polling average** — the governing party's national share in the **April** monthly average from *PollBase* (already in the repo at `westminster_opposition_leads/PollBase-latest.xlsx`), i.e. the month before the early-May elections. This matches LocalBase's own "poll rating in advance" figures. PollBase's monthly-average tab is a simple unweighted mean of that month's polls, so for 2026 — not yet on that tab — we take the same unweighted mean of the individual April 2026 polls from the raw `24-` sheet (5 polls). April 1982 is missing entirely, so that single year falls back to LocalBase's poll rating.
- **Governing party** is taken as the party in office on polling day (so 1997 and 2024 are Conservative; 2010 is Labour, who governed until polling day; 2025 and 2026 are Labour).

Note on methodology: using pure BBC PNS, the governing party underperformed its polling average in **34 of 44** election-years since 1982. Stack's headline "33 of 43" (to 2025) came from averaging the BBC and Thrasher-Rallings shares; the two measures are within one year of each other.

In [9]:
POLLING_DIR = REPO_ROOT / "source_data" / "polling"
POLLING_DIR.mkdir(parents=True, exist_ok=True)

LOCALBASE_URL = "https://www.markpack.org.uk/files/2026/05/LocalBase-local-elections-results.xlsx"
LOCALBASE_PATH = POLLING_DIR / "localbase_local_elections_results.xlsx"
POLLBASE_PATH = REPO_ROOT / "westminster_opposition_leads" / "PollBase-latest.xlsx"

PNS_VS_POLLS_CSV = NOTEBOOK_DIR / "governing_party_pns_vs_polls.csv"
PNS_VS_POLLS_PNG = NOTEBOOK_DIR / "governing_party_pns_vs_polls.png"

FIRST_YEAR, LAST_YEAR = 1982, 2026


def download_bytes_if_missing(url: str, path: Path) -> Path:
    if path.exists():
        return path
    response = requests.get(url, headers=REQUEST_HEADERS, timeout=300)
    response.raise_for_status()
    path.write_bytes(response.content)
    return path


def governing_party(year: int) -> str:
    """Party in office on local-election polling day (early May)."""
    if year <= 1997:
        return "Conservative"  # Thatcher / Major (1997 locals were on GE day, still Conservative going in)
    if year <= 2010:
        return "Labour"  # Blair / Brown (Brown governed until polling day in 2010)
    if year <= 2024:
        return "Conservative"  # Cameron coalition through Sunak (2024 locals pre-July GE)
    return "Labour"  # Starmer (2025, 2026)


download_bytes_if_missing(LOCALBASE_URL, LOCALBASE_PATH)

# LocalBase 'Results' sheet has a two-row header; pull the BBC PNS and poll-rating columns by position.
localbase_raw = pd.read_excel(LOCALBASE_PATH, sheet_name="Results", header=None)
LOCALBASE_COLS = {
    "year": 0,
    "Conservative_pns": 4, "Labour_pns": 9, "LD_pns": 16,   # BBC Projected National Share
    "Conservative_pollrating": 34, "Labour_pollrating": 35,  # LocalBase 'poll rating in advance' (1982 fallback)
}
localbase = localbase_raw.iloc[4:][list(LOCALBASE_COLS.values())].copy()
localbase.columns = list(LOCALBASE_COLS.keys())
for column in localbase.columns:
    localbase[column] = pd.to_numeric(localbase[column], errors="coerce")
localbase = localbase[localbase["year"].between(FIRST_YEAR, LAST_YEAR)].set_index("year")

# PollBase April monthly averages = the governing party's polling average going into the May elections.
pollbase_monthly = pd.read_excel(POLLBASE_PATH, sheet_name="Monthly average", header=0)[
    ["Date", "Conservative", "Labour", "LD"]
]
pollbase_monthly["Date"] = pd.to_datetime(pollbase_monthly["Date"], errors="coerce")
pollbase_monthly = pollbase_monthly.dropna(subset=["Date"])
april_polls = pollbase_monthly[pollbase_monthly["Date"].dt.month == 4].copy()
april_polls["year"] = april_polls["Date"].dt.year
april_polls = april_polls.groupby("year")[["Conservative", "Labour", "LD"]].mean()

# Where the monthly-average tab is not yet updated (e.g. 2026), fall back to the simple unweighted
# mean of the individual April polls from the raw recent-polls sheet. (Verified: PollBase's monthly
# average is itself the unweighted mean of that month's polls, so this is the same calculation.)
recent_polls = pd.read_excel(POLLBASE_PATH, sheet_name="24-", header=0)[["Year", "Month", "Con", "Lab", "LD"]]
recent_polls["Year"] = pd.to_numeric(recent_polls["Year"], errors="coerce").ffill()
recent_polls["Month"] = recent_polls["Month"].ffill()
recent_april = recent_polls[recent_polls["Month"] == "Apr"].copy()
for column in ["Con", "Lab", "LD"]:
    recent_april[column] = pd.to_numeric(recent_april[column], errors="coerce")
recent_april = recent_april.groupby("Year")[["Con", "Lab", "LD"]].mean().rename(
    columns={"Con": "Conservative", "Lab": "Labour"}
)

records = []
for year in localbase.index:
    party = governing_party(int(year))
    pns = localbase.loc[year, f"{party}_pns"]
    if year in april_polls.index and pd.notna(april_polls.loc[year, party]):
        poll_average = april_polls.loc[year, party]               # PollBase monthly-average tab (1983-2025)
    elif year in recent_april.index and pd.notna(recent_april.loc[year, party]):
        poll_average = recent_april.loc[year, party]              # raw April polls (2026)
    else:
        poll_average = localbase.loc[year, f"{party}_pollrating"]  # 1982 fallback
    records.append(
        {"year": int(year), "governing_party": party, "pns": pns, "poll_average": poll_average, "pns_minus_poll": pns - poll_average}
    )

pns_vs_polls = pd.DataFrame(records).dropna(subset=["pns_minus_poll"])
pns_vs_polls.to_csv(PNS_VS_POLLS_CSV, index=False)

n_underperformed = int((pns_vs_polls["pns_minus_poll"] < 0).sum())
n_total = len(pns_vs_polls)
display(pns_vs_polls)
print(f"Governing party underperformed its polling average in {n_underperformed} of {n_total} election-years.")

# --- chart, in the same house style as the other charts in this notebook ---
LABOUR_RED = "#E4003B"
CONSERVATIVE_BLUE = "#0087DC"

sns.set_theme(style="white")
fig, ax = plt.subplots(figsize=(12, 7))

bar_colors = [LABOUR_RED if party == "Labour" else CONSERVATIVE_BLUE for party in pns_vs_polls["governing_party"]]
ax.bar(pns_vs_polls["year"], pns_vs_polls["pns_minus_poll"], color=bar_colors, width=0.8)
ax.axhline(0, color="#333333", lw=1)

fig.suptitle(
    "Governing parties almost always perform worse at local\nelections than their polling averages",
    x=0.01, y=0.99, ha="left", fontsize=18, fontweight="bold",
)
fig.text(
    0.01, 0.90,
    f"Governing party's BBC Projected National Share minus its April polling average, by local-election year. "
    f"The party in power underperformed the polls in {n_underperformed} of {n_total} years since {FIRST_YEAR}.",
    ha="left", va="top", fontsize=12, color="#555555",
)

ax.set_ylabel("PNS minus polling average (percentage points)", fontsize=13, fontweight="bold")
ax.set_xlabel("")
ax.yaxis.set_major_formatter(lambda value, _: f"{value:+.0f}".replace("+0", "0"))
ax.set_xticks(range(FIRST_YEAR, LAST_YEAR + 1, 2))
ax.set_xticklabels(range(FIRST_YEAR, LAST_YEAR + 1, 2), rotation=45, ha="right")
ax.tick_params(axis="both", labelsize=11, length=0)
for label in ax.get_xticklabels() + ax.get_yticklabels():
    label.set_fontweight("bold")
ax.grid(axis="y", color="#E6E6E6", lw=0.8)
ax.set_axisbelow(True)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

legend = ax.legend(
    handles=[Patch(facecolor=LABOUR_RED, label="Labour"), Patch(facecolor=CONSERVATIVE_BLUE, label="Conservative")],
    title="Governing party",
    loc="lower left", frameon=False, fontsize=11, title_fontsize=11, ncol=2,
)

fig.text(
    0.01, 0.01,
    "Sources: BBC Projected National Share via Mark Pack's LocalBase; governing party's April national polling average from PollBase.",
    ha="left", fontsize=9, color="#555555",
)

plt.tight_layout(rect=[0, 0.05, 1, 0.88])
fig.savefig(PNS_VS_POLLS_PNG, dpi=200, bbox_inches="tight")
plt.close(fig)
print(f"Saved chart to {PNS_VS_POLLS_PNG}")

C:\Users\DamayantiChatterjee\AppData\Local\Programs\Python\Python314\Lib\site-packages\openpyxl\worksheet\header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


C:\Users\DamayantiChatterjee\AppData\Local\Programs\Python\Python314\Lib\site-packages\openpyxl\worksheet\header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


C:\Users\DamayantiChatterjee\AppData\Local\Programs\Python\Python314\Lib\site-packages\openpyxl\worksheet\header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


,year,governing_party,pns,poll_average,pns_minus_poll
0,1982,Conservative,40.0,35.700000,4.300000
1,1983,Conservative,40.5,44.300000,-3.800000
2,1984,Conservative,37.0,40.666667,-3.666667
3,1985,Conservative,32.0,36.225000,-4.225000
4,1986,Conservative,34.0,32.260000,1.740000
5,1987,Conservative,40.0,41.035714,-1.035714
6,1988,Conservative,40.0,43.300000,-3.300000
7,1989,Conservative,38.0,41.575000,-3.575000
8,1990,Conservative,32.0,31.007692,0.992308
9,1991,Conservative,35.0,40.950000,-5.950000


Governing party underperformed its polling average in 34 of 44 election-years.


Saved chart to C:\Users\DamayantiChatterjee\GitHub\uk-politics\labour_lessons_2026_elections\governing_party_pns_vs_polls.png


## Councils up for election in May 2027, by current control

May 2027 is the biggest year in the electoral cycle. This map shows every council holding a local election in May 2027, shaded by the party in control **after the May 2026 results** (Open Council Data UK).

What's included:
- **All 32 Scottish councils** and **all 22 Welsh councils** — both nations elect their whole councils every five years (last 2022, next 2027).
- **England's metropolitan boroughs that elect in 2027** — the 32 boroughs that elect by thirds (identified as those that also contested 2023, which repeats in 2027). The four all-out metropolitan boroughs whose cycle skips 2027 are excluded: Birmingham (2026→2030), Doncaster and Rotherham, and St Helens.

What's excluded:
- **Northern Ireland** — its 11 councils also vote in 2027, but they are power-sharing with no single controlling party, so there is no "holding party" to colour them by.
- **England's reorganising shire areas** — county/district areas being abolished into new unitary authorities for 2027 have no published ward geometry yet and no post-2026 control, so they cannot be placed on this map.

Minority administrations are coloured by the party running the council (e.g. "SNP minority" → SNP); councils run by a multi-party coalition are shown as Coalition. Non-contesting GB councils are drawn in light grey for context.

In [10]:
COUNCILS_2027_MAP_PATH = NOTEBOOK_DIR / "councils_up_2027_by_control.png"
COUNCILS_2027_CSV = NOTEBOOK_DIR / "councils_up_2027_by_control.csv"

# Colours for council-control groups (a superset of the ward map's palette, adding Coalition / TBC).
CONTROL_COLORS = {
    "Labour": "#E4003B",
    "Conservative": "#0087DC",
    "Liberal Democrat": "#FAA61A",
    "SNP": "#FDF38E",
    "Plaid Cymru": "#3F8428",
    "Green": "#6AB023",
    "Reform UK": "#12B6CF",
    "Independent": "#7F8C8D",
    "Coalition": "#6F42C1",
    "Other / NOC": "#B0A8B9",
}
CONTROL_LEGEND_ORDER = list(CONTROL_COLORS.keys())
BACKGROUND_FILL = "#EFEFEF"
BACKGROUND_EDGE = "#FFFFFF"

# 2026 control by council (reuse the lookup built earlier; blank/"NOC" falls into "Other / NOC").
control_by_council = {}
for council_key, group in zip(council_control_2026["council_key"], council_control_2026["control_group"]):
    control_by_council[council_key] = "Other / NOC" if group in (None, "Other", "TBC") or pd.isna(group) else group

# English metropolitan boroughs that elect in 2027 = E08 councils that also contested 2023 (thirds cycle repeats).
councils_2023 = set(races.loc[races["year"] == 2023, "council"].map(normalise_council))


def up_in_2027(lad_code: str, council_key: str) -> bool:
    if lad_code.startswith(("S", "W")):  # all Scottish and Welsh councils
        return True
    if lad_code.startswith("E08") and council_key in councils_2023:  # metropolitan boroughs on the thirds cycle
        return True
    return False


# Walk the boundary file once: light-grey GB backdrop, and 2027 councils coloured by control.
background_patches = []
patches_by_group = {group: [] for group in CONTROL_LEGEND_ORDER}
councils_shown = {}
for feature in boundary_features:
    code = str(feature["properties"]["WD25CD"])
    if code.startswith("N"):  # exclude Northern Ireland
        continue
    patches = geometry_to_patches(feature["geometry"])
    background_patches.extend(patches)
    council_key = normalise_council(feature["properties"]["LAD25NM"])
    if up_in_2027(feature["properties"]["LAD25CD"], council_key):
        group = control_by_council.get(council_key, "Other / NOC")
        patches_by_group.setdefault(group, []).extend(patches)
        councils_shown[council_key] = (feature["properties"]["LAD25NM"], group)

councils_table = (
    pd.DataFrame([(nm, grp) for nm, grp in councils_shown.values()], columns=["council", "control_2026"])
    .sort_values(["control_2026", "council"]).reset_index(drop=True)
)
councils_table.to_csv(COUNCILS_2027_CSV, index=False)
n_councils = len(councils_table)
display(councils_table["control_2026"].value_counts().rename("councils"))

fig, ax = plt.subplots(figsize=(11, 14))
ax.add_collection(
    PatchCollection(background_patches, facecolor=BACKGROUND_FILL, edgecolor=BACKGROUND_EDGE, linewidths=0.05, zorder=1)
)
for group in CONTROL_LEGEND_ORDER:
    patches = patches_by_group.get(group, [])
    if not patches:
        continue
    ax.add_collection(
        PatchCollection(patches, facecolor=CONTROL_COLORS[group], edgecolor="white", linewidths=0.1, zorder=2)
    )

ax.autoscale_view()
ax.set_aspect("equal")
ax.axis("off")

fig.suptitle(
    "Councils holding local elections in May 2027, by current control",
    x=0.01, y=0.985, ha="left", fontsize=17, fontweight="bold",
)
fig.text(
    0.01, 0.955,
    f"All {n_councils} councils up for election in May 2027, shaded by the party in control after the 2026 results: "
    "every Scottish and Welsh council plus England's metropolitan boroughs. Northern Ireland (no single-party control) is not shown.",
    ha="left", va="top", fontsize=11, color="#555555", wrap=True,
)

legend_handles = [
    Patch(facecolor=CONTROL_COLORS[group], edgecolor="none", label=group)
    for group in CONTROL_LEGEND_ORDER
    if patches_by_group.get(group)
]
ax.legend(handles=legend_handles, loc="lower left", bbox_to_anchor=(0.01, -0.02), ncol=2, frameon=False, fontsize=11)

fig.text(
    0.01, 0.015,
    "Sources: council control from Open Council Data UK (position after the May 2026 elections); 2027 cycle per the standard electoral timetable; "
    "ONS Wards (December 2025) Boundaries UK BGC. Reorganising English shire areas and Northern Ireland are excluded.",
    ha="left", fontsize=8.5, color="#555555", wrap=True,
)

plt.tight_layout(rect=[0, 0.04, 1, 0.93])
fig.savefig(COUNCILS_2027_MAP_PATH, dpi=200, bbox_inches="tight")
plt.close(fig)
print(f"Saved chart to {COUNCILS_2027_MAP_PATH} ({n_councils} councils)")

control_2026
Labour              34
Coalition           17
SNP                 10
Reform UK            8
Other / NOC          5
Independent          4
Plaid Cymru          4
Conservative         3
Liberal Democrat     1
Name: councils, dtype: int64

Saved chart to C:\Users\DamayantiChatterjee\GitHub\uk-politics\labour_lessons_2026_elections\councils_up_2027_by_control.png (86 councils)


### Full 2027 footprint (provisional version)

The map above is deliberately conservative — it only shows councils that can be placed and coloured with confidence. This second version trades accuracy for completeness: it adds **every English district and unitary on the 2023→2027 cycle**, shaded by its control after the 2026 results on its **current (December 2025) boundaries**.

**Health warning — this is provisional for much of England.** Many of the shire areas shown here are mid-reorganisation: their district/county councils are due to be abolished and replaced by **new unitary authorities** in 2027. So for those areas this map shows the *outgoing* councils on *old* boundaries, not the bodies that will actually be elected. Treat the English shire areas as an approximate geographic footprint of where 2027 elections will happen, not as the final list of councils. Scotland and Wales are unchanged from the map above; Northern Ireland is still excluded.

In [11]:
COUNCILS_2027_FULL_MAP_PATH = NOTEBOOK_DIR / "councils_up_2027_by_control_full_footprint.png"
COUNCILS_2027_FULL_CSV = NOTEBOOK_DIR / "councils_up_2027_by_control_full_footprint.csv"


def up_in_2027_full(lad_code: str, council_key: str) -> bool:
    """Lower-fidelity footprint: all Scottish/Welsh councils plus every English council on the
    2023->2027 cycle (districts and unitaries included), on current boundaries. Provisional where
    the underlying councils are being reorganised into new unitary authorities."""
    if lad_code.startswith(("S", "W")):
        return True
    if lad_code.startswith("E") and council_key in councils_2023:
        return True
    return False


background_patches = []
patches_by_group = {group: [] for group in CONTROL_LEGEND_ORDER}
councils_shown = {}
for feature in boundary_features:
    code = str(feature["properties"]["WD25CD"])
    if code.startswith("N"):  # exclude Northern Ireland
        continue
    patches = geometry_to_patches(feature["geometry"])
    background_patches.extend(patches)
    council_key = normalise_council(feature["properties"]["LAD25NM"])
    if up_in_2027_full(feature["properties"]["LAD25CD"], council_key):
        group = control_by_council.get(council_key, "Other / NOC")
        patches_by_group.setdefault(group, []).extend(patches)
        councils_shown[council_key] = (feature["properties"]["LAD25NM"], group)

councils_table_full = (
    pd.DataFrame([(nm, grp) for nm, grp in councils_shown.values()], columns=["council", "control_2026"])
    .sort_values(["control_2026", "council"]).reset_index(drop=True)
)
councils_table_full.to_csv(COUNCILS_2027_FULL_CSV, index=False)
n_full = len(councils_table_full)
display(councils_table_full["control_2026"].value_counts().rename("councils"))

fig, ax = plt.subplots(figsize=(11, 14))
ax.add_collection(
    PatchCollection(background_patches, facecolor=BACKGROUND_FILL, edgecolor=BACKGROUND_EDGE, linewidths=0.05, zorder=1)
)
for group in CONTROL_LEGEND_ORDER:
    patches = patches_by_group.get(group, [])
    if not patches:
        continue
    ax.add_collection(
        PatchCollection(patches, facecolor=CONTROL_COLORS[group], edgecolor="white", linewidths=0.08, zorder=2)
    )
ax.autoscale_view()
ax.set_aspect("equal")
ax.axis("off")

fig.suptitle(
    "Councils up in May 2027 - full footprint (provisional for England)",
    x=0.01, y=0.985, ha="left", fontsize=17, fontweight="bold",
)
fig.text(
    0.01, 0.955,
    f"All {n_full} councils on the 2027 cycle, shaded by control after the 2026 results. Scotland and Wales as before; "
    "English shire districts and unitaries added on their current (pre-reorganisation) boundaries. Northern Ireland excluded.",
    ha="left", va="top", fontsize=11, color="#555555", wrap=True,
)
legend_handles = [
    Patch(facecolor=CONTROL_COLORS[group], edgecolor="none", label=group)
    for group in CONTROL_LEGEND_ORDER if patches_by_group.get(group)
]
ax.legend(handles=legend_handles, loc="lower left", bbox_to_anchor=(0.01, -0.02), ncol=2, frameon=False, fontsize=11)
fig.text(
    0.01, 0.015,
    "Provisional: many English shire areas shown here are due to be abolished and replaced by new unitary authorities in 2027; "
    "they appear on outgoing boundaries with 2026 control. Sources: Open Council Data UK; ONS Wards (December 2025) Boundaries UK BGC.",
    ha="left", fontsize=8.5, color="#555555", wrap=True,
)
plt.tight_layout(rect=[0, 0.04, 1, 0.93])
fig.savefig(COUNCILS_2027_FULL_MAP_PATH, dpi=200, bbox_inches="tight")
plt.close(fig)
print(f"Saved chart to {COUNCILS_2027_FULL_MAP_PATH} ({n_full} councils)")

control_2026
Labour              84
Coalition           70
Conservative        38
Liberal Democrat    34
Other / NOC         18
Independent         13
SNP                 10
Reform UK            9
Plaid Cymru          4
Green                3
Name: councils, dtype: int64

Saved chart to C:\Users\DamayantiChatterjee\GitHub\uk-politics\labour_lessons_2026_elections\councils_up_2027_by_control_full_footprint.png (283 councils)
